# Re-run: graded example answers (Paper VI, RQ3)

This notebook recomputes the follow-up table of RQ3 and the figures the text derives from it, from the
per-case model outputs published with release `ai-grader-reliability-2026` and nothing else: no model
call, no corpus, no repository code. It needs Python 3.9+ with numpy, and `per-case-model-outputs.jsonl`
(or the `.gz` the report offers) in the same folder as this notebook, or in a folder named by
`RELEASE_DIR`.

Each answer was graded under six arms that differ only in the three example answers shown with it:
none (control), the examples without marks (ungraded), with the teacher's marks (correct), with those
three marks deranged among the examples (shuffled), and with every mark moved two points up or down,
clipped to 0–10 (shiftUp, shiftDown). Run `exmarks/lite` is Gemini 3.5 Flash Lite on 600 answers;
`exmarks/gpt41mini` is GPT-4.1-mini on the first 150. Each case carries the marks each arm showed
(`shown`) and the model's mark in each arm (`scores`).

**Bias** is the mean of grader minus teacher; **MAE** the mean absolute error; **QWK** quadratic
weighted kappa with both marks rounded half up onto 0..10. The contrasts are differences of arm
biases on the same answers.

In [1]:
# Finding the published data. This cell is the same in every notebook of this report.
import gzip, json, os, sys
from pathlib import Path

# Importing from the release's code/ folder must not leave __pycache__ folders in it.
sys.dont_write_bytecode = True

RELEASE_ID = "ai-grader-reliability-2026"   # set when the report is built
DATA_NAME = "per-case-model-outputs.jsonl"


def find_release_file(name=DATA_NAME):
    """Look beside this notebook, in RELEASE_DIR, in the folders above, and in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    places += [here, *here.parents]
    places += [p / "research/wiki/research/releases" / RELEASE_ID for p in [here, *here.parents]]
    for d in places:
        for n in (name, name + ".gz"):
            if (d / n).is_file():
                return d / n
    raise FileNotFoundError(
        f"Could not find {name} (or {name}.gz). Put it in the same folder as this notebook: download it "
        f"from the report ('Download the data file', beside each notebook) or take it from the release "
        f"directory. Or set the environment variable RELEASE_DIR to the folder that holds it.")


def per_case(runs=None):
    """Yield the per-case records of the published file, optionally only for the named runs."""
    path = find_release_file()
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            if r.get("record") == "case" and (runs is None or r["run"] in runs):
                yield r


def code_dir():
    """The release's code package (its code/ folder): in RELEASE_DIR, beside this notebook or in a folder
    above it (as when the notebook sits in the unzipped release's notebooks/ folder), in a folder named
    after the release there, or in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    for p in [here, *here.parents]:
        places += [p, p / RELEASE_ID, p / "research/wiki/research/releases" / RELEASE_ID]
    for d in places:
        if (d / "code" / "SHA256SUMS").is_file():
            return d / "code"
    raise FileNotFoundError(
        f"Could not find the code package of release {RELEASE_ID} (its code/ folder). Download the release "
        f"from the report ('Download the release', in the sidebar or beside each notebook), unzip it, and "
        f"run this notebook from the unzipped folder or its notebooks/ folder. Or set the environment "
        f"variable RELEASE_DIR to the unzipped {RELEASE_ID} folder.")


def find_code_file(relpath):
    """A file inside the release's code package, by its path there (e.g. studies/.../out/result.json)."""
    path = code_dir() / relpath
    if not path.is_file():
        raise FileNotFoundError(f"The code package at {code_dir()} has no {relpath}: is it release {RELEASE_ID}?")
    return path


def load_code_json(relpath):
    """A committed output of the code package, parsed."""
    return json.loads(find_code_file(relpath).read_text(encoding="utf-8"))


def shown(path):
    """A path as printed: relative to this notebook's folder, so an executed notebook names no machine's folders."""
    return os.path.relpath(path, Path.cwd())


def corpus_file(env_var, name):
    """A public corpus file the reader has downloaded: $env_var/name, or name beside this notebook."""
    for d in ([Path(os.environ[env_var])] if os.environ.get(env_var) else []) + [Path.cwd()]:
        if (d / name).is_file():
            return d / name
    raise FileNotFoundError(f"This notebook also needs {name} from the public corpus. Download the corpus "
                            f"(its location and hash are in the release file), then set {env_var} to its "
                            f"folder or put {name} beside this notebook.")


print("Data:", shown(find_release_file()))

Data: per-case-model-outputs.jsonl


In [2]:
import hashlib
import numpy as np

ARMS = ["control", "ungraded", "correct", "shuffled", "shiftUp", "shiftDown"]

class Run:
    def __init__(self, run):
        rows = [r for r in per_case({run}) if all(r["scores"][a] is not None for a in ARMS)]
        self.model = rows[0]["model"]
        self.ids = [r["id"] for r in rows]
        self.h = np.array([r["human"] for r in rows], float)
        self.s = {a: np.array([r["scores"][a] for r in rows], float) for a in ARMS}
        self.shown = {a: np.array([np.mean(r["shown"][a]) for r in rows]) for a in ("shiftUp", "shiftDown")}
        self.q = np.unique([r["questionId"] for r in rows], return_inverse=True)[1]
        self.l = np.unique([str(r["learnerId"]) for r in rows], return_inverse=True)[1]
        self.n = len(rows)

lite, gpt = Run("exmarks/lite"), Run("exmarks/gpt41mini")
print(f"{lite.model}: {lite.n} answers complete in all six arms; {gpt.model}: {gpt.n}")

google:gemini-3.5-flash-lite: 600 answers complete in all six arms; openai:gpt-4.1-mini: 146


In [3]:
def arm_stats(R, W):
    """Bias, MAE and QWK per arm for each row of case weights W (row 0 = every case once).
    QWK in closed form on integer marks: 1 - sum w (h-m)^2 / (S2h + S2m - 2 S1h S1m / N)."""
    N = W.sum(1)
    hi = np.clip(np.floor(R.h + 0.5), 0, 10)
    out = {}
    for a in ARMS:
        s = R.s[a]
        mi = np.clip(np.floor(s + 0.5), 0, 10)
        den = W @ hi ** 2 + W @ mi ** 2 - 2 * (W @ hi) * (W @ mi) / N
        out[a] = {"bias": W @ (s - R.h) / N, "mae": W @ np.abs(s - R.h) / N, "qwk": 1 - (W @ (hi - mi) ** 2) / den}
    out["shown"] = (W @ R.shown["shiftUp"] - W @ R.shown["shiftDown"]) / N
    return out

def derived(st):
    b = {a: st[a]["bias"] for a in ARMS}
    return {
        "dCorr": b["correct"] - b["control"],           # what correct marks do, against no examples
        "dUngr": b["ungraded"] - b["control"],          # examples without marks, against no examples
        "R": b["shuffled"] - b["correct"],              # wrong pairing, same marks, same level
        "rho": (b["shuffled"] - b["control"]) / (b["correct"] - b["control"]),   # share the shuffle keeps
        "S": b["shiftUp"] - b["shiftDown"],             # grade moved by the shown level
        "U": b["shiftUp"] - b["correct"], "D": b["shiftDown"] - b["correct"],
        "kappa": (b["shiftUp"] - b["shiftDown"]) / st["shown"],   # points of grade per point of shown level
        "shown": st["shown"],
        "dMAE_D": st["shiftDown"]["mae"] - st["correct"]["mae"], "dQWK_D": st["shiftDown"]["qwk"] - st["correct"]["qwk"],
        "dQWK_ungr": st["ungraded"]["qwk"] - st["control"]["qwk"], "dQWK_corr": st["correct"]["qwk"] - st["control"]["qwk"],
    }

one = lambda R: np.ones((1, R.n))   # noqa: E731
P = arm_stats(lite, one(lite))
D = {k: float(v[0]) for k, v in derived(P).items()}
print(f"{'Arm (Gemini, n = 600)':12s} {'bias':>6s} {'MAE':>5s} {'QWK':>6s}")
for a in ARMS:
    print(f"{a:12s} {P[a]['bias'][0]:+6.2f} {P[a]['mae'][0]:5.2f} {P[a]['qwk'][0]:6.3f}")
print({k: round(v, 3) for k, v in D.items()})

Arm (Gemini, n = 600)   bias   MAE    QWK
control       +0.19  1.21  0.854
ungraded      +0.61  1.24  0.835
correct       -0.15  1.19  0.857
shuffled      +0.06  1.20  0.848
shiftUp       +0.33  1.23  0.844
shiftDown     -0.80  1.44  0.814
{'dCorr': -0.343, 'dUngr': 0.417, 'R': 0.215, 'rho': 0.372, 'S': 1.127, 'U': 0.478, 'D': -0.649, 'kappa': 0.387, 'shown': 2.917, 'dMAE_D': 0.241, 'dQWK_D': -0.043, 'dQWK_ungr': -0.018, 'dQWK_corr': 0.003}


In [4]:
# The paper's printed values. Point estimates must match exactly, as printed: compared as formatted
# text, because a mean such as 723/600 = 1.205 sits on a rounding tie that round() and format() break
# differently.
def same(x, printed, nd):
    return f"{x:.{nd}f}" == f"{printed:.{nd}f}"

PRINTED = {"control": (0.19, 1.21, 0.854), "ungraded": (0.61, 1.24, 0.835), "correct": (-0.15, 1.19, 0.857),
           "shuffled": (0.06, 1.20, 0.848), "shiftUp": (0.33, 1.23, 0.844), "shiftDown": (-0.80, 1.44, 0.814)}
for a, (bi, mae, k) in PRINTED.items():
    got = (P[a]["bias"][0], P[a]["mae"][0], P[a]["qwk"][0])
    assert same(got[0], bi, 2) and same(got[1], mae, 2) and same(got[2], k, 3), (a, got, (bi, mae, k))
TEXT = {"dCorr": (-0.34, 2), "R": (0.22, 2), "rho": (0.37, 2), "S": (1.13, 2), "shown": (2.92, 2),
        "kappa": (0.39, 2), "U": (0.48, 2), "D": (-0.65, 2), "dUngr": (0.42, 2),
        "dMAE_D": (0.24, 2), "dQWK_D": (-0.043, 3), "dQWK_ungr": (-0.018, 3)}
for k, (v, nd) in TEXT.items():
    assert same(D[k], v, nd), (k, D[k], v)
print("All 18 table values and 12 values in the text match the paper.")

All 18 table values and 12 values in the text match the paper.


**The original 150.** The follow-up's first 150 answers are the original run's draw. The text says the
follow-up reproduced every cell of the original table (control +0.38 / 1.24 / 0.840; graded examples
+0.06 / 1.22 / 0.840; marks removed +0.63 / 1.26 / 0.821) within 0.06, with the control +0.32 on the
same answers. The original run's own per-case output is not in the release, so its table is not
recomputed here; the comparison is.

In [5]:
first = np.isin(lite.ids, lite.ids[:150])
P150 = arm_stats(lite, first[None, :].astype(float))
ORIGINAL = {"control": (0.38, 1.24, 0.840), "correct": (0.06, 1.22, 0.840), "ungraded": (0.63, 1.26, 0.821)}
gap = max(abs(P150[a][k][0] - v) for a, vals in ORIGINAL.items() for k, v in zip(("bias", "mae", "qwk"), vals))
for a in ORIGINAL:
    print(f"first 150, {a:9s} bias {P150[a]['bias'][0]:+.2f}  MAE {P150[a]['mae'][0]:.2f}  QWK {P150[a]['qwk'][0]:.3f}")
assert same(P150["control"]["bias"][0], 0.32, 2) and gap <= 0.06, gap
print(f"largest difference from the original table: {gap:.3f} (paper: within 0.06)")

first 150, control   bias +0.32  MAE 1.23  QWK 0.841
first 150, correct   bias +0.00  MAE 1.17  QWK 0.846
first 150, ungraded  bias +0.67  MAE 1.28  QWK 0.814
largest difference from the original table: 0.060 (paper: within 0.06)


**Intervals.** The study resamples questions (the three examples belong to the question): 4,000
cluster-bootstrap resamples, percentile 95% intervals, from a generator seeded by the SHA-256 of
`"real|<model id>"` (`grader-exemplar-marks/analyse.py`, `weights` and `intervals`). The cell below
uses the same scheme, seed and order, so on the same numpy version its endpoints are the study's;
the printed intervals are compared, not asserted, because another numpy version can draw a different
random stream.

In [6]:
def seed_of(key):
    return int.from_bytes(hashlib.sha256(key.encode()).digest()[:4], "little")

def question_boot(R, reps=4000):
    rng = np.random.default_rng(seed_of(f"real|{R.model}"))
    G = R.q.max() + 1
    draw = rng.integers(0, G, size=(reps, G))
    counts = np.zeros((reps, G))
    np.add.at(counts, (np.repeat(np.arange(reps), G), draw.ravel()), 1)
    st = arm_stats(R, counts[:, R.q])
    ci = {a: np.percentile(st[a]["bias"], [2.5, 97.5]) for a in ARMS}
    with np.errstate(divide="ignore", invalid="ignore"):
        ci.update({k: np.nanpercentile(v, [2.5, 97.5]) for k, v in derived(st).items()})
    return ci

ci = question_boot(lite)
for a in ARMS:
    print(f"{a:10s} bias {P[a]['bias'][0]:+.2f} [{ci[a][0]:+.2f}, {ci[a][1]:+.2f}]")
for k, lab, f in (("dCorr", "correct − control", "{:+.2f}"), ("R", "shuffled − correct", "{:+.2f}"),
                  ("rho", "share the shuffle keeps", "{:.0%}"), ("kappa", "slope, grade per shown point", "{:.2f}"),
                  ("dUngr", "ungraded − control", "{:+.2f}")):
    print(f"{lab:30s} {f.format(D[k])} [{f.format(ci[k][0])}, {f.format(ci[k][1])}]")
print("paper: correct − control −0.34 [−0.48, −0.21]; shuffled − correct 0.22 [0.12, 0.31]; "
      "share 37% [2%, 60%]; slope 0.39 [0.35, 0.42]; ungraded − control +0.42 [+0.29, +0.55]")

control    bias +0.19 [-0.01, +0.41]
ungraded   bias +0.61 [+0.43, +0.79]
correct    bias -0.15 [-0.34, +0.06]
shuffled   bias +0.06 [-0.11, +0.26]
shiftUp    bias +0.33 [+0.15, +0.52]
shiftDown  bias -0.80 [-1.00, -0.57]
correct − control              -0.34 [-0.48, -0.21]
shuffled − correct             +0.22 [+0.12, +0.31]
share the shuffle keeps        37% [2%, 60%]
slope, grade per shown point   0.39 [0.35, 0.42]
ungraded − control             +0.42 [+0.29, +0.55]
paper: correct − control −0.34 [−0.48, −0.21]; shuffled − correct 0.22 [0.12, 0.31]; share 37% [2%, 60%]; slope 0.39 [0.35, 0.42]; ungraded − control +0.42 [+0.29, +0.55]


**The second model.** GPT-4.1-mini on 146 of the first 150 answers: control bias −0.77, correct
examples −0.06 (−0.17 to +0.03), QWK +0.033 (0.005 to 0.068), and a slope of 0.16 points per point.

In [7]:
Pg = arm_stats(gpt, one(gpt))
Dg = {k: float(v[0]) for k, v in derived(Pg).items()}
assert gpt.n == 146
assert same(Pg["control"]["bias"][0], -0.77, 2) and same(Dg["dCorr"], -0.06, 2)
assert same(Dg["dQWK_corr"], 0.033, 3) and same(Dg["kappa"], 0.16, 2)
cg = question_boot(gpt)
print(f"GPT-4.1-mini: control bias {Pg['control']['bias'][0]:+.2f}; correct − control {Dg['dCorr']:+.2f} "
      f"[{cg['dCorr'][0]:+.2f}, {cg['dCorr'][1]:+.2f}]; dQWK {Dg['dQWK_corr']:+.3f} "
      f"[{cg['dQWK_corr'][0]:+.3f}, {cg['dQWK_corr'][1]:+.3f}]; slope {Dg['kappa']:.2f}")

GPT-4.1-mini: control bias -0.77; correct − control -0.06 [-0.17, +0.03]; dQWK +0.033 [+0.005, +0.068]; slope 0.16
